In [7]:
from pathlib import Path
import pickle
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eda

In [8]:
def load_items(stimuli_dir: Path, trial_id: int) -> np.ndarray:
    csv_path = stimuli_dir / f"civ_items_trial_{trial_id}.csv"
    df = pd.read_csv(csv_path)
    numeric = df.drop(columns=["Name"], errors="ignore")
    return numeric.to_numpy(dtype=np.int64)

def get_eda_params(items: np.ndarray) -> dict:
    n_obj = items.shape[1] - 1

    if n_obj == 3:
        n_selected = 6
        max_row_diff = 5
    elif n_obj == 5:
        n_selected = 10
        max_row_diff = 500
    else:
        raise ValueError(f"Number of objectives {n_obj} not supported")

    return {
        "n_items": items.shape[0],
        "n_obj": n_obj,
        "n_con": 1,
        "n_selected": n_selected,
        "capacity": n_selected * 10,
        "pop_size": 1_000,
        "generations": 100,
        "max_no_improve_gen": 5,
        "max_row_diff": max_row_diff,
    }

def aspi_to_p_rank(items: np.ndarray, n_obj: int, aspi_item: np.ndarray, temp: float = 0.3) -> np.ndarray:
    aspi_item = np.asarray(aspi_item, dtype=float)
    aspi_unit = aspi_item / (np.linalg.norm(aspi_item))
    item_scores = items[:, :n_obj] @ aspi_unit
    ranks = item_scores.argsort().argsort().astype(float)
    scaled = ranks / (ranks.max() + 1e-12)
    logits = scaled / temp
    logits -= logits.max()
    p_rank = np.exp(logits)
    p_rank /= p_rank.sum()
    return p_rank

def run_eda_pass(items: np.ndarray, params: dict, p_rank: np.ndarray, seed: int) -> dict:
    eda_process = eda.KnapsackEDA(
        items=items,
        capacity=params["capacity"],
        n_selected=params["n_selected"],
        n_obj=params["n_obj"],
        pop_size=params["pop_size"],
        generations=params["generations"],
        max_no_improve_gen=params["max_no_improve_gen"],
        max_row_diff=params["max_row_diff"],
        seed=seed,
        p_rank=p_rank,
    )
    return eda_process.run()

def save_pass_results(run_name: str, results: dict, output_dir: Path, use_human_input: bool = True) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    result_type = "eda_human" if use_human_input else "eda"
    file_path = output_dir / f"{result_type}_{run_name}.pkl"

    with open(file_path, "wb") as f:
        pickle.dump(results, f)

    return file_path

In [9]:
def gen_aspi(ref_sol, pf_actual):
    # normalize by 95 percentile
    # p95_list = [np.percentile(pf_actual[:, i], 95) for i in range(5)]
    # aspi = ref_sol/p95_list

    # normalize by z-score
    # aspi = (ref_sol - np.mean(pf_actual, axis=0)) / np.std(pf_actual, axis=0)

    # min-max quantile
    q5 = np.percentile(pf_actual, 5, axis=0)
    q95 = np.percentile(pf_actual, 95, axis=0)
    aspi = (ref_sol - q5) / (q95 - q5 + 1e-12)
    return aspi

# test different ways of normalizing the aspiration vector
# plot original and normalized aspiration vectors and pf
# check if the pf generated matches the original aspiration vector 
# (e.g. [50, 50, 50, 75, 75] percentiles generate pf with solutions around these percentiles))
# transform all to normalized scale

def select_test_ref(trial_id: int) -> np.ndarray:
    with open(f'card_game/eda_results/eda_trial{trial_id}.pkl', 'rb') as f:
        results = pickle.load(f)
    pf_actual = results['converged_pf_table'][-1]
    ref_sol = np.median(pf_actual, axis=0)
    # ref_sol = np.array([np.percentile(pf_actual[:, i], 75) for i in range(5)])
    # p25_list = [np.percentile(pf_actual[:, i], 25) for i in range(5)]
    # p75_list = [np.percentile(pf_actual[:, i], 75) for i in range(5)]
    # ref_sol = np.array([p25_list[0], p25_list[1], p25_list[2], p75_list[3], p75_list[4]])
    return ref_sol, pf_actual

In [10]:
# obtain item list
trial_id = 8
stimuli_dir = Path("card_game/stimuli")
items = load_items(stimuli_dir=stimuli_dir, trial_id=trial_id)

# obtain params
params = get_eda_params(items)
n_obj = params["n_obj"]
temp = 0.3

# generate aspiration vector and probabilities
ref_sol, pf_actual = select_test_ref(trial_id)
aspi = gen_aspi(ref_sol, pf_actual)
p_rank = aspi_to_p_rank(items, n_obj, aspi, temp=temp)

In [13]:
# check normalization
contribution = pf_actual * aspi
mean_contribution = contribution.mean(axis=0)
print(mean_contribution / mean_contribution.sum())

from scipy.stats import spearmanr, kendalltau
score_true = pf_actual @ ref_sol
score_norm = pf_actual @ aspi
print(spearmanr(score_true, score_norm))
print(kendalltau(score_true, score_norm))

best_true = np.argmax(score_true)
best_norm = np.argmax(score_norm)

print(best_true, best_norm)
print(pf_actual[best_true])
print(pf_actual[best_norm])

print(ref_sol)
print(aspi)

ref_prop = ref_sol / ref_sol.sum()
aspi_prop = aspi / aspi.sum()

print(ref_prop)
print(aspi_prop)
print("L1 distance:", np.sum(np.abs(ref_prop - aspi_prop)))

[0.17733336 0.16241863 0.23767577 0.2162635  0.20630875]
SignificanceResult(statistic=0.8751550912443539, pvalue=0.0)
SignificanceResult(statistic=0.6914242621122302, pvalue=0.0)
915 3544
[ 61  60 130 106 106]
[ 72  76 121  99 100]
[ 83.  73. 105.  98.  92.]
[0.5        0.52380952 0.53191489 0.52083333 0.53061224]
[0.18403548 0.16186253 0.23281596 0.2172949  0.20399113]
[0.19177883 0.20091115 0.20402003 0.19976961 0.20352039]
L1 distance: 0.0935839437343303


In [11]:
# run EDA
run_name = "minmax_quant_median"
results = run_eda_pass(
    items=items,
    params=params,
    p_rank=p_rank,
    seed=1123,
)

# save results
output_dir = Path("data/eda_results")
save_path = save_pass_results(run_name, results, output_dir, use_human_input=(p_rank is not None))
pf = results["converged_pf_table"][-1]

# save info
file_path = output_dir / f"history_{run_name}.pkl"
history = {
    "trial_id": trial_id,
    "original_aspi": ref_sol,
    "normalized_aspi": aspi,
    "temp": temp,
    "p_rank": p_rank,
}
with open(file_path, "wb") as f:
    pickle.dump(history, f)

In [ ]:
run_name = "minmax_quant_median"
with open(f"data/eda_results/eda_human_{run_name}.pkl", "rb") as f:
    results = pickle.load(f)
pf = results["converged_pf_table"][-1]

with open(f"data/eda_results/history_{run_name}.pkl", "rb") as f:
    history = pickle.load(f)
    ref_sol = history["original_aspi"]
    aspi = history["normalized_aspi"]

objective_pairs = list(itertools.combinations(range(5), 2))  # all 10 pairs
fig, axes = plt.subplots(2, 5, figsize=(24, 8))
axes = axes.ravel()
for ax, (a, b) in zip(axes, objective_pairs):
    ax.plot(pf[:, a], pf[:, b], "bo", alpha=0.2, markersize=3, label="PF")
    ax.plot(
        ref_sol[a], ref_sol[b],
        "rs", alpha=1, markersize=6, label="Reference"
    )
    # ax.plot(
    #     aspi[a], aspi[b],
    #     "gs", alpha=1, markersize=6, label="Aspiration"
    # )
    ax.set_xlabel(f"Obj {a + 1}")
    ax.set_ylabel(f"Obj {b + 1}")
    ax.set_xlim(30, 150)
    ax.set_ylim(40, 150)
fig.tight_layout()
plt.show()